# End-to-End Pairs Trading System

This notebook demonstrates the end-to-end pipeline for the Statistical, Machine Learning, and Deep Learning driven Pairs Trading system.

In [ ]:
import sys
import os
sys.path.append(os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src import data_loader, pair_selection, spread_model, ml_filter, dl_model, backtest, explainability

# Set plot style
plt.style.use('seaborn-v0_8-darkgrid')

## 1. Data Loading and Preprocessing
Using NIFTY 50 large-cap tickers.

In [ ]:
TICKERS = [
    'RELIANCE.NS', 'TCS.NS', 'HDFCBANK.NS', 'ICICIBANK.NS', 'INFY.NS',
    'ITC.NS', 'SBIN.NS', 'BHARTIARTL.NS', 'LT.NS', 'BAJFINANCE.NS',
    'HUL.NS', 'AXISBANK.NS', 'KOTAKBANK.NS', 'M&M.NS', 'TATAMOTORS.NS'
]

prices = data_loader.download_data(TICKERS, start_date='2018-01-01')
log_prices, returns = data_loader.preprocess_data(prices)
features = data_loader.calculate_features(returns)

## 2. Pair Selection (Clustering + Cointegration)

In [ ]:
clustered_features = pair_selection.find_clusters(features, n_clusters=3, method='agglomerative')
valid_pairs = pair_selection.select_cointegrated_pairs(log_prices, clustered_features, p_value_threshold=0.05)
print(f"Selected Pairs: {valid_pairs}")

## 3. Dynamic Spread, Signal Generation, and ML Trade Filtering

In [ ]:
X_all, y_all = [], []
pair_features = {}

# Use the first valid pair for demonstration
if len(valid_pairs) > 0:
    target_pair = valid_pairs[0]
    print(f"Analyzing Pair: {target_pair}")
    
    spread, z, beta = spread_model.compute_spread_and_zscore(log_prices, target_pair)
    positions = spread_model.generate_positions(z)
    
    # ML Feature Extraction
    X, y, valid_indices = ml_filter.extract_ml_features_and_labels(spread, z)
    
    # Walk-forward Split
    X_train, X_test, y_train, y_test = ml_filter.walk_forward_validation(X, y)
    
    # Train Random Forest with Isotonic Calibration
    ml_prob, clf = ml_filter.train_and_predict_rf(X_train, y_train, X_test, y_test)
else:
    print("No valid cointegrated pairs found in the given dataset.")

## 4. Regime Detection and Backtesting
Calculates mark-to-market continuous daily PnL and strict percentage drawdown.

In [ ]:
if len(valid_pairs) > 0:
    vol_regime = backtest.identify_volatility_regime(returns)
    position_size = backtest.compute_position_sizes(ml_prob, vol_regime, valid_indices, len(z))
    
    # PnL Calculation (Continuous Mark-to-Market)
    pnl = backtest.calculate_pair_pnl(spread, positions, position_size)
    
    # Calculate trade-level metrics and format output
    metrics = backtest.calculate_metrics(pnl, positions)
    backtest.print_performance_dashboard(metrics)
    
    # Plotting
    backtest.plot_performance(metrics)

## 5. Explainability (SHAP)

In [ ]:
if len(valid_pairs) > 0:
    feature_names = ['zscore', 'abs_zscore', 'spread_vol', 'spread_momentum']
    explainer, shap_vals = explainability.generate_shap_explanations(clf, X_test, feature_names)

## 6. Deep Learning (LSTM) Auxiliary Model

In [ ]:
if len(valid_pairs) > 0:
    # Prepare sequence data
    X_dl, y_dl = dl_model.prepare_dl_sequences(spread, seq_length=20)
    split = int(0.7 * len(X_dl))
    X_train_dl, X_test_dl = X_dl[:split], X_dl[split:]
    y_train_dl, y_test_dl = y_dl[:split], y_dl[split:]
    
    # Train LSTM
    lstm_model, device = dl_model.train_lstm(X_train_dl, y_train_dl, epochs=20)
    
    # Predict
    dl_preds = dl_model.predict_lstm(lstm_model, X_test_dl, device)
    
    plt.figure(figsize=(10, 4))
    plt.plot(y_test_dl[:100], label='True Spread (t+5)')
    plt.plot(dl_preds[:100], label='LSTM Predicted Spread')
    plt.title("LSTM Spread Prediction (First 100 out-of-sample points)")
    plt.legend()
    plt.show()